In [ ]:
##Best Model with best Parameter 


import pandas as pd
import numpy as np
import re

from collections import Counter, defaultdict
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# ============================================================
# PATHS
# ============================================================
DEV_PATH  = "/kaggle/input/news-article-v4/development.csv"
EVAL_PATH = "/kaggle/input/news-article-v4/evaluation.csv"
SUB_PATH  = "/kaggle/working/submission_tuned.csv"

# ============================================================
# BEST PARAMS
# ============================================================
MIN_RULE_SUPPORT = 34
MIN_RULE_PURITY  = 0.926328564964768

WORD_NG_MAX = 2
CHAR_NG_MAX = 5
MIN_DF      = 2
MAX_DF      = 0.8782583211530898
C_VALUE     = 0.64491922705094

# ============================================================
# LOAD DATA
# ============================================================
df_dev  = pd.read_csv(DEV_PATH)
df_eval = pd.read_csv(EVAL_PATH)

for df in (df_dev, df_eval):
	df["article"] = df["article"].fillna("").astype(str)
	df["title"]   = df["title"].fillna("").astype(str)
	df["source"]  = df["source"].fillna("").astype(str)

# ============================================================
# TEXT + NUMERIC FEATURES
# ============================================================
def build_model_text(df):
	return (df["title"] + " " + df["article"]).str.lower()

df_dev["text"]  = build_model_text(df_dev)
df_eval["text"] = build_model_text(df_eval)

def add_numeric(df):
	df["n_tokens"]    = df["article"].str.split().str.len()
	df["title_len"]   = df["title"].str.len()
	df["article_len"] = df["article"].str.len()
	df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)
	return df

df_dev  = add_numeric(df_dev)
df_eval = add_numeric(df_eval)

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]

for df in (df_dev, df_eval):
	df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], 0).fillna(0)

FEATURES = ["source", "text"] + NUM_COLS

X_dev  = df_dev[FEATURES]
y_dev  = df_dev["label"].astype(int)
X_eval = df_eval[FEATURES]

# ============================================================
# RULE MINING
# ============================================================
def tokenize_for_rules(text):
	return re.findall(r"[a-z0-9_:/\.]+", text.lower())

def mine_pure_rules(texts, labels):
	counts = defaultdict(lambda: Counter())

	for txt, y in zip(texts, labels):
		for tok in set(tokenize_for_rules(txt)):
			counts[tok][int(y)] += 1

	rule_token_to_class = {}
	rule_meta = {}

	for tok, c in counts.items():
		total = sum(c.values())
		if total < MIN_RULE_SUPPORT:
			continue

		best_class, best_freq = c.most_common(1)[0]
		purity = best_freq / total

		if purity >= MIN_RULE_PURITY:
			rule_token_to_class[tok] = int(best_class)
			rule_meta[tok] = (purity, total)

	return rule_token_to_class, rule_meta

def apply_rules(texts, rule_token_to_class, rule_meta):
	rule_pred = np.full(len(texts), -1, dtype=int)

	for i, txt in enumerate(texts):
		toks = set(tokenize_for_rules(txt))
		hits = [t for t in toks if t in rule_token_to_class]
		if not hits:
			continue

		hits.sort(
			key=lambda t: (rule_meta[t][0], rule_meta[t][1]),
			reverse=True
		)
		rule_pred[i] = rule_token_to_class[hits[0]]

	return rule_pred

# ============================================================
# BASELINE MODEL
# ============================================================
def make_model():
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w_tfidf", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1, WORD_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=250_000
			), "text"),
			("c_tfidf", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, CHAR_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
			("num", StandardScaler(), NUM_COLS),
		],
		remainder="drop",
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C_VALUE,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([
		("pre", pre),
		("clf", clf),
	])

# ============================================================
# TRAIN ON FULL DEV
# ============================================================
model = make_model()
model.fit(X_dev, y_dev)

rule_token_to_class, rule_meta = mine_pure_rules(
	df_dev["article"],
	y_dev
)

print("Rules mined:", len(rule_token_to_class))

# ============================================================
# PREDICT ON EVAL (TWO-STAGE)
# ============================================================
model_pred = model.predict(X_eval)

rule_pred = apply_rules(
	df_eval["article"],
	rule_token_to_class,
	rule_meta
)

final_pred = model_pred.copy()
mask = rule_pred != -1
final_pred[mask] = rule_pred[mask]

print(f"Rule coverage on eval: {mask.mean():.4f}")

# ============================================================
# SUBMISSION
# ============================================================
submission = pd.DataFrame({
	"Id": df_eval["Id"].astype(int),
	"Predicted": final_pred.astype(int)
})

submission.to_csv(SUB_PATH, index=False)
print("Saved submission to:", SUB_PATH)
